# Travel Capstone — Hotel Recommendation System

**Project Summary:** This notebook builds an **item-based collaborative filtering** recommender from `hotels.csv` (40,552 stays, 1,340 users, 9 hotels). Stays (nights booked) are treated as implicit ratings; hotel–hotel cosine similarity drives personalised suggestions, with a popularity model as cold-start fallback. The trained artifact powers the Streamlit web app in the repo.

**GitHub Repo:** https://github.com/<your-username>/travel-mlops-capstone

## 1. Problem Statement

Given a user's booking history, suggest hotels they are likely to enjoy. With no explicit star ratings available, we use **implicit feedback**: the more nights a user has spent in a hotel, the stronger their revealed preference. This is the standard approach when interaction logs exist but ratings do not.

In [ ]:
import pandas as pd
import numpy as np

hotels = pd.read_csv('hotels.csv')
users = pd.read_csv('users.csv')
print(hotels.shape)
print('Hotels:', hotels['name'].nunique(), '| Places:', hotels['place'].nunique(),
      '| Users:', hotels['userCode'].nunique())
hotels.head()

## 2. EDA — Popularity, Price and Coverage

We examine which hotels dominate bookings and how prices vary. Reasoning: popularity statistics double as our cold-start recommender for brand-new users, and price context lets the Streamlit app filter suggestions by budget.

In [ ]:
popularity = (hotels.groupby('name')
              .agg(stays=('days', 'count'), total_nights=('days', 'sum'),
                   avg_price=('price', 'mean'), place=('place', 'first'))
              .sort_values('stays', ascending=False))
popularity

## 3. Build the User–Item Matrix

Rows = users, columns = hotels, values = total nights stayed. This matrix is the core data structure of collaborative filtering: two hotels whose columns look alike are visited by the same kinds of travellers.

In [ ]:
user_item = hotels.pivot_table(index='userCode', columns='name',
                               values='days', aggfunc='sum').fillna(0)
print(user_item.shape)
user_item.head()

## 4. Item-Based Collaborative Filtering

We compute **cosine similarity between hotel columns**. Item-based (rather than user-based) CF is chosen because the item space is tiny (9 hotels) and stable, making the similarity matrix cheap to compute, robust, and instantly usable at inference — a deliberate engineering trade-off for deployment.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

item_sim = pd.DataFrame(cosine_similarity(user_item.T),
                        index=user_item.columns, columns=user_item.columns)
item_sim.round(3)

In [ ]:
def recommend(user_code, top_n=3):
    """Score = similarity-weighted sum of the user's past stays; unseen hotels only."""
    if user_code not in user_item.index:
        return popularity.head(top_n).index.tolist()   # cold start → most popular
    history = user_item.loc[user_code]
    scores = item_sim.mul(history, axis=0).sum(axis=0)
    scores = scores[history == 0]                      # exclude already-visited
    if scores.empty:
        return popularity.head(top_n).index.tolist()
    return scores.nlargest(top_n).index.tolist()

for uc in [0, 5, 99999]:
    print(f'User {uc}: {recommend(uc)}')

## 5. Validation — Leave-Last-Out Hit Rate

We hide each active user's most-stayed hotel, recommend from the rest of their history, and measure how often the hidden hotel appears in the top-3. This simulates the real question — "would we have suggested the hotel the user actually chose?" — and gives an honest offline metric for an implicit-feedback system.

In [ ]:
hits = trials = 0
for uc in user_item.index:
    row = user_item.loc[uc]
    visited = row[row > 0]
    if len(visited) < 2:
        continue
    held_out = visited.idxmax()
    masked = row.copy(); masked[held_out] = 0
    scores = item_sim.mul(masked, axis=0).sum(axis=0)
    scores = scores[masked == 0]
    trials += 1
    hits += held_out in scores.nlargest(3).index
print(f'Top-3 hit rate: {hits/trials:.2%}  over {trials} users')

## 6. Save Artifact for the Streamlit App

In [ ]:
import joblib
joblib.dump({'user_item': user_item, 'item_sim': item_sim, 'popularity': popularity},
            'hotel_recommender.pkl', compress=3)
print('Recommender artifact saved.')

## 7. Project Summary

- Converted 40k stay records into an implicit-feedback user–item matrix.
- Item-based cosine CF with popularity fallback handles both returning and cold-start users.
- Evaluated with leave-last-out top-3 hit rate; artifact consumed by the Streamlit dashboard (`streamlit_app/app.py`).

**GitHub Repo:** https://github.com/<your-username>/travel-mlops-capstone